# 🔍 Detección de Fraude en Transacciones Financieras

**Dataset:** Credit Card Fraud Detection (Kaggle) — 284,807 transacciones reales europeas  
**Problema:** Clasificación binaria extremadamente desbalanceada (fraude = 0.17% de transacciones)  

**Por qué este problema es crítico en fintech:**
- Yape, Tigo Money y cualquier billetera digital procesan millones de transacciones diarias
- Un modelo que diga siempre 'no fraude' tiene 99.83% de accuracy → completamente inútil
- El costo asimétrico (perder $500 vs rechazar una transacción legítima) requiere métricas especiales

**Pipeline:**
1. EDA del desbalance extremo
2. Técnicas de balanceo: SMOTE vs undersampling
3. Isolation Forest (detección de anomalías no supervisada)
4. Random Forest + XGBoost (supervisado)
5. Métricas correctas: AUC-PR, F1, costo de negocio
6. Umbral óptimo por costo asimétrico

In [ ]:
# ─── DEPENDENCIAS ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report, f1_score
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
SEED = 42
print('✅ Librerías cargadas')

## 1. Carga y Comprensión del Problema

In [ ]:
# ─── CARGA ───────────────────────────────────────────────────────────────────
# Descarga desde: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
# Archivo: creditcard.csv → colocar en datos/
df = pd.read_csv('datos/creditcard.csv')

print('Shape:', df.shape)
print('\nColumnas:')
print('  V1-V28: componentes PCA (anonimizadas por privacidad)')
print('  Time:   segundos desde primera transacción')
print('  Amount: monto de la transacción')
print('  Class:  0=legítima, 1=fraude')

fraudes = df['Class'].sum()
total   = len(df)
print(f'\n⚠️  DESBALANCE EXTREMO:')
print(f'   Legítimas: {total - fraudes:,} ({(total-fraudes)/total*100:.2f}%)')
print(f'   Fraudes:   {fraudes:,} ({fraudes/total*100:.4f}%)')
print(f'\n   Ratio: 1 fraude cada {int(total/fraudes)} transacciones')

In [ ]:
# ─── EDA: DISTRIBUCIÓN DE MONTOS ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Desbalance
counts = df['Class'].value_counts()
axes[0].bar(['Legítima', 'Fraude'], counts.values,
            color=['#2E86C1', '#E74C3C'], alpha=0.85, edgecolor='white')
axes[0].set_title('Distribución de Clases\n(escala lineal)', fontweight='bold')
axes[0].set_ylabel('Transacciones')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Monto por clase
df[df['Class']==0]['Amount'].clip(upper=500).hist(
    ax=axes[1], bins=60, alpha=0.6, color='#2E86C1', density=True, label='Legítima')
df[df['Class']==1]['Amount'].clip(upper=500).hist(
    ax=axes[1], bins=60, alpha=0.6, color='#E74C3C', density=True, label='Fraude')
axes[1].set_title('Distribución de Montos\n(recortado en $500)', fontweight='bold')
axes[1].set_xlabel('Monto ($)')
axes[1].legend()

# Fraudes por hora del día
df['hora'] = (df['Time'] % 86400) / 3600
fraudes_hora = df[df['Class']==1].groupby(df[df['Class']==1]['hora'].astype(int)).size()
fraudes_hora.plot(kind='bar', ax=axes[2], color='#E74C3C', alpha=0.8)
axes[2].set_title('Fraudes por Hora del Día', fontweight='bold')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('N° fraudes')

plt.tight_layout()
plt.savefig('img/01_eda_fraude.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n💡 Monto promedio transacción legítima: ${df[df["Class"]==0]["Amount"].mean():.2f}')
print(f'   Monto promedio fraude:               ${df[df["Class"]==1]["Amount"].mean():.2f}')

## 2. Preprocesamiento

In [ ]:
# ─── FEATURES Y SPLIT ────────────────────────────────────────────────────────
# Escalar Amount y Time (las únicas no-PCA)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

FEATURES = [c for c in df.columns if c.startswith('V')] + ['Amount_scaled', 'Time_scaled']
X = df[FEATURES]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Fraudes en test: {y_test.sum()} ({y_test.mean()*100:.3f}%)')

## 3. Isolation Forest — Detección de Anomalías (No Supervisado)

In [ ]:
# ─── ISOLATION FOREST ────────────────────────────────────────────────────────
# Técnica no supervisada: no necesita etiquetas durante entrenamiento
# Útil cuando los fraudes son tan raros que el modelo supervisado lucha
# Idea: las anomalías son más fáciles de 'aislar' en un árbol aleatorio

print('Entrenando Isolation Forest...')
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=fraudes/total,  # Le damos la tasa real de fraude
    random_state=SEED,
    n_jobs=-1
)
iso_forest.fit(X_train)

# -1 = anomalía, 1 = normal → convertir a 0/1
pred_iso = (iso_forest.predict(X_test) == -1).astype(int)
scores_iso = -iso_forest.score_samples(X_test)  # Mayor score = más anómalo

auc_iso = roc_auc_score(y_test, scores_iso)
ap_iso  = average_precision_score(y_test, scores_iso)

print(f'\nIsolation Forest:')
print(f'  AUC-ROC:           {auc_iso:.4f}')
print(f'  Average Precision: {ap_iso:.4f}')
print(f'\n{classification_report(y_test, pred_iso, target_names=["Legítima", "Fraude"])}')

## 4. Random Forest con SMOTE — Supervisado + Balanceo

In [ ]:
# ─── SMOTE + RANDOM FOREST ────────────────────────────────────────────────────
# SMOTE: Synthetic Minority Oversampling Technique
# Genera ejemplos sintéticos de fraude interpolando entre ejemplos reales
# Mejor que duplicar ejemplos porque crea variabilidad

modelos_supervisados = {
    'RF + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
        ('model', RandomForestClassifier(
            n_estimators=200, max_depth=10,
            min_samples_leaf=10, random_state=SEED, n_jobs=-1
        ))
    ]),
    'RF + Undersampling': ImbPipeline([
        ('under', RandomUnderSampler(random_state=SEED, sampling_strategy=0.1)),
        ('model', RandomForestClassifier(
            n_estimators=200, max_depth=10,
            class_weight='balanced', random_state=SEED, n_jobs=-1
        ))
    ]),
    'LR + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('model', LogisticRegression(max_iter=1000, random_state=SEED))
    ]),
}

resultados = {}
for nombre, pipeline in modelos_supervisados.items():
    print(f'Entrenando {nombre}...')
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    ap    = average_precision_score(y_test, proba)
    resultados[nombre] = {'pipeline': pipeline, 'proba': proba, 'auc': auc, 'ap': ap}
    print(f'  AUC-ROC: {auc:.4f} | Avg Precision: {ap:.4f}')

mejor_nombre = max(resultados, key=lambda k: resultados[k]['ap'])
print(f'\n🏆 Mejor modelo (por AP): {mejor_nombre}')

In [ ]:
# ─── COMPARACIÓN VISUAL ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colores = ['#2E86C1', '#E74C3C', '#27AE60', '#8E44AD']

# Todos los modelos incluyendo Isolation Forest
todos = {**resultados, 'Isolation Forest': {'proba': scores_iso, 'auc': auc_iso, 'ap': ap_iso}}

for (nombre, res), color in zip(todos.items(), colores):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC={res["auc"]:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{nombre} (AP={res["ap"]:.3f})')

axes[0].plot([0,1],[0,1],'k--',lw=1); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC', fontweight='bold'); axes[0].legend(fontsize=9)

axes[1].axhline(y_test.mean(), color='gray', linestyle='--', lw=1, label=f'Baseline ({y_test.mean():.4f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall\n(más informativa con desbalance extremo)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('img/02_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Umbral Óptimo por Costo de Negocio

In [ ]:
# ─── COSTO ASIMÉTRICO ────────────────────────────────────────────────────────
# En detección de fraude:
#   FN (fraude no detectado)  → pérdida del monto + costo reputacional
#   FP (legítima rechazada)   → cliente insatisfecho, costo de atención
#
# Estimación conservadora:
#   Costo FN = monto promedio del fraude en test
#   Costo FP = $2 (costo de atención al cliente)

mejor = resultados[mejor_nombre]
montos_test = df.loc[y_test.index, 'Amount'].values

COSTO_FP = 2.0
umbrales  = np.arange(0.01, 0.99, 0.005)
costos    = []

for umbral in umbrales:
    pred = (mejor['proba'] >= umbral).astype(int)
    cm   = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    # FN: perdemos el monto promedio de los fraudes no detectados
    monto_fn = montos_test[(y_test == 1) & (pred == 0)].mean() if fn > 0 else 0
    costo = fn * monto_fn + fp * COSTO_FP
    costos.append(costo)

umbral_optimo = umbrales[np.argmin(costos)]
pred_opt      = (mejor['proba'] >= umbral_optimo).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(umbrales, costos, color='#E74C3C', lw=2)
axes[0].axvline(umbral_optimo, color='#2E86C1', linestyle='--', lw=2,
                label=f'Umbral óptimo: {umbral_optimo:.3f}')
axes[0].set_xlabel('Umbral'); axes[0].set_ylabel('Costo total ($)')
axes[0].set_title('Costo total por umbral', fontweight='bold'); axes[0].legend()

cm_opt = confusion_matrix(y_test, pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Pred: Legítima', 'Pred: Fraude'],
            yticklabels=['Real: Legítima', 'Real: Fraude'])
axes[1].set_title(f'Confusión — Umbral {umbral_optimo:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('img/03_umbral_costo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🎯 Umbral óptimo: {umbral_optimo:.3f}')
print(f'\n{classification_report(y_test, pred_opt, target_names=["Legítima", "Fraude"])}')

## 6. Feature Importance y Perfiles de Fraude

In [ ]:
# ─── FEATURE IMPORTANCE ──────────────────────────────────────────────────────
rf_model = resultados[mejor_nombre]['pipeline'].named_steps['model']
importancias = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
top_features = importancias.tail(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_features.plot(kind='barh', ax=axes[0], color='#2E86C1', alpha=0.85)
axes[0].set_title('Top 15 Features más importantes\n(Random Forest)', fontweight='bold')
axes[0].set_xlabel('Importancia relativa')

# Distribución de top feature por clase
top_feat = top_features.index[-1]
df[df['Class']==0][top_feat].clip(-5,5).hist(
    ax=axes[1], bins=60, alpha=0.6, color='#2E86C1', density=True, label='Legítima')
df[df['Class']==1][top_feat].clip(-5,5).hist(
    ax=axes[1], bins=60, alpha=0.6, color='#E74C3C', density=True, label='Fraude')
axes[1].set_title(f'Distribución de {top_feat}\npor clase (recortado -5 a 5)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/04_features_fraude.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Conclusión: Las variables PCA con mayor poder discriminativo son las que')
print('   tienen distribuciones más separadas entre fraude y transacción legítima.')

In [ ]:
# ─── RESUMEN FINAL ────────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN — DETECCIÓN DE FRAUDE')
print('=' * 60)
print(f'\nDataset: {total:,} transacciones | {fraudes} fraudes ({fraudes/total*100:.3f}%)')
print(f'\nComparación de modelos:')
print(f'{"Modelo":<25} {"AUC-ROC":>10} {"Avg Prec":>10}')
print('-' * 47)
print(f'{"Isolation Forest":<25} {auc_iso:>10.4f} {ap_iso:>10.4f}')
for nombre, res in resultados.items():
    print(f'{nombre:<25} {res["auc"]:>10.4f} {res["ap"]:>10.4f}')
print(f'\n🏆 Mejor modelo: {mejor_nombre}')
print(f'   Umbral óptimo: {umbral_optimo:.3f}')
print(f'\n⚠️  Lección clave: con 0.17% de fraudes, Avg Precision importa más que AUC-ROC')
print(f'   Un modelo aleatorio tiene AUC=0.5 pero también puede tener AUC alto si el')
print(f'   desbalance es extremo. La curva Precision-Recall es más honesta.')